# EU Conventions Tree Traversal Examples

This notebook shows how to use the LATTICE LLM-guided traversal flow on the EU conventions semantic tree.

It is intentionally example-oriented:
- no gold paths
- no evaluation metrics
- no tests
- just load a tree, run example queries, inspect the traversal, and visualize the prediction tree

Before running the traversal cells, make sure the API key for your selected LLM backend is available in the environment.


In [3]:
from __future__ import annotations

import os
import pickle
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from google.genai import types as genai_types

from hyperparams import HyperParams
from llm_apis import GenAIAPI, OpenAIResponsesAPI, VllmAPI
from prompts import get_traversal_prompt_response_constraint
from tree_construction.build_llm_bottom_up_tree import run_coro_sync
from tree_objects import InferSample, SemanticNode
from utils import compute_node_registry, get_all_leaf_nodes_with_path, post_process, setup_logger, visualize_sample

REPO_ROOT


c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happen

WindowsPath('C:/Users/jmgil/Desktop/TESE/LATTICE/llm-guided-retrieval')

In [4]:
# Configuration
#
# TREE_PATH prefers the expected EU conventions tree path. The fallback points to
# the tree path produced by the current bottom-up construction notebook in this repo.
default_tree_path = REPO_ROOT / "trees" / "EU" / "eu_conventions_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"
if not default_tree_path.exists():
    default_tree_path = REPO_ROOT / "trees" / "EU" / "codigo_civil_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"

# HyperParams.from_args(...) mirrors the way run.py configures retrieval.
# We pass the required args first, then override/add notebook-specific values.
hp = HyperParams.from_args("--subset fiqa --tree_version eu_conventions_notebook")
hp.TREE_PATH = str(default_tree_path)
hp.DATASET = "EU"
hp.LLM_API_BACKEND = "openai"  # one of: openai, genai, vllm
hp.LLM = "gpt-4.1"
hp.LLM_API_TIMEOUT = 120
hp.LLM_API_MAX_RETRIES = 4
hp.LLM_MAX_CONCURRENT_CALLS = 4
hp.LLM_API_STAGGERING_DELAY = 0.05
hp.VLLM_BASE_URL = "http://localhost:8000/v1"
hp.REASONING_IN_TRAVERSAL_PROMPT = -1
hp.SUBSET = "fiqa"  # chooses the traversal relevance definition
hp.MAX_BEAM_SIZE = 8
hp.SEARCH_WITH_PATH_RELEVANCE = True
hp.NUM_LEAF_CALIB = 0
hp.RELEVANCE_CHAIN_FACTOR = 0.5
hp.MAX_PROMPT_PROTO_SIZE = None
hp.MAX_DOC_DESC_CHAR_LEN = None

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("eu_conventions_traversal_notebook", str(LOG_DIR / "eu_conventions_traversal_notebook.log"))

hp


Log file already exists: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\eu_conventions_traversal_notebook.log, appending to it.


S=fiqa-TV=eu_conventions_notebook-TPV=5-RInTP=-1-NumLC=10-PlTau=5.0-RCF=0.5-LlmApiB=openai-Llm=gpt-4.1-NumI=20-NumES=1000-MaxBS=2-TP=C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\EU\eu_conventions_notebook\eu_conventions_tree-bottom-up-llm.pkl-LlmApiB=openai-Llm=gpt-4.1-VBUrl=http:____localhost:8000__v1-RInTP=-1-S=fiqa-MaxBS=8-NumLC=0-RCF=0.5

In [5]:
tree_path = Path(hp.TREE_PATH)
tree_obj = pickle.loads(tree_path.read_bytes())
semantic_root_node = SemanticNode().load_dict(tree_obj) if isinstance(tree_obj, dict) else tree_obj
node_registry = compute_node_registry(semantic_root_node)
all_leaf_nodes = get_all_leaf_nodes_with_path(semantic_root_node)

print(f"Loaded tree from: {tree_path}")
print(f"Total nodes in semantic tree: {len(node_registry)}")
print(f"Total leaf nodes: {len(all_leaf_nodes)}")
print("Root description preview:")
print(semantic_root_node.desc[:800])


Loaded tree from: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\EU\eu_conventions_notebook\eu_conventions_tree-bottom-up-llm.pkl
Total nodes in semantic tree: 556
Total leaf nodes: 361
Root description preview:
ROOT Node: This subtree addresses questions about the foundational principles, legal procedures, and organizational structure of the European Union and the European Convention on Human Rights (ECHR). It covers the ECHR's objectives and relationship to broader human rights frameworks, the process for signing EU treaties, the structure and governance of EU institutions, and the rights, procedures, and committees established by EU Protocols. Use this subtree for queries on EU treaty signing, institutional roles, legislative processes, protected rights and freedoms, emergency derogations, committee mechanisms, and family law protections under EU law.. Key topics: ECHR foundational principles; EU treaty signing procedures; EU institutions and governance; EU legislati

In [6]:
def make_llm_api(hp, logger):
    if hp.LLM_API_BACKEND == "genai":
        llm_api = GenAIAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "openai":
        llm_api = OpenAIResponsesAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "vllm":
        llm_api = VllmAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, base_url=hp.VLLM_BASE_URL)
    else:
        raise ValueError(f"Unknown backend: {hp.LLM_API_BACKEND}")

    llm_api_kwargs = {
        "max_concurrent_calls": hp.LLM_MAX_CONCURRENT_CALLS,
        "response_mime_type": "application/json",
        "response_schema": get_traversal_prompt_response_constraint(bool(hp.REASONING_IN_TRAVERSAL_PROMPT)),
        "staggering_delay": hp.LLM_API_STAGGERING_DELAY,
        "print_summary_report": False,
        "thinking_config": genai_types.ThinkingConfig(thinking_budget=hp.REASONING_IN_TRAVERSAL_PROMPT),
    }

    if hp.LLM_API_BACKEND == "vllm":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")
        llm_api_kwargs.pop("response_schema")

    if hp.LLM_API_BACKEND == "openai":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")

    return llm_api, llm_api_kwargs


llm_api, llm_api_kwargs = make_llm_api(hp, logger)
print(type(llm_api).__name__)


2026-05-08 17:14:24,229 - eu_conventions_traversal_notebook - INFO - Initialized client for model: gpt-4.1
2026-05-08 17:14:24,634 - eu_conventions_traversal_notebook - INFO - Initialized OpenAI Responses client with model: gpt-4.1


OpenAIResponsesAPI


## Example Queries

Pick one of these or write your own EU Convention query in the next cell.


In [7]:
EXAMPLE_QUERIES = [
    "What does Article 6 say about the right to a fair trial?",
    "When can the right to liberty and security be restricted?",
    "What protections does Article 8 give for private and family life?",
    "What does the Convention say about prohibition of torture?",
    "How does Protocol 1 protect property rights?",
    "What are the rules for individual applications to the European Court of Human Rights?",
]

for idx, query in enumerate(EXAMPLE_QUERIES):
    print(f"[{idx}] {query}")


[0] What does Article 6 say about the right to a fair trial?
[1] When can the right to liberty and security be restricted?
[2] What protections does Article 8 give for private and family life?
[3] What does the Convention say about prohibition of torture?
[4] How does Protocol 1 protect property rights?
[5] What are the rules for individual applications to the European Court of Human Rights?


In [8]:
query = EXAMPLE_QUERIES[0]
num_iters = 12
query


'What does Article 6 say about the right to a fair trial?'

In [9]:
def make_sample(query: str) -> InferSample:
    return InferSample(
        semantic_root_node,
        node_registry,
        hp=hp,
        logger=logger,
        query=query,
        gold_paths=[],
        excluded_ids_set=set(),
    )


def run_one_iteration(sample: InferSample):
    inputs = sample.get_step_prompts()
    prompts = [prompt for prompt, _ in inputs]
    slates = [slate for _, slate in inputs]

    raw_responses = run_coro_sync(llm_api.run_batch(prompts, **llm_api_kwargs))
    response_jsons = [post_process(output, return_json=True) for output in raw_responses]
    sample.update(slates, response_jsons)
    return raw_responses, response_jsons


def print_frontier(sample: InferSample):
    print("Current beam state paths:")
    for state_path in sample.beam_state_paths:
        node = state_path[-1]
        print(f"  path={node.path} | path_relevance={node.path_relevance:.3f} | desc={node.desc[:180]}")


def print_top_predictions(sample: InferSample, k: int = 5):
    top_preds = sample.get_top_predictions(k=k)
    if not top_preds:
        print("No leaf predictions yet. Increase num_iters.")
        return

    for rank, (node, score) in enumerate(top_preds, start=1):
        print(f"[{rank}] path={node.path} | score={score:.3f} | id={node.id}")
        print(node.desc[:600])
        print("-" * 120)


def run_query(query: str, num_iters: int = 4) -> InferSample:
    sample = make_sample(query)
    print(f"Running traversal for query: {query}")
    for step in range(num_iters):
        print(f"\n--- Iteration {step + 1} ---")
        _, response_jsons = run_one_iteration(sample)
        print_frontier(sample)
        if response_jsons:
            first_reasoning = response_jsons[0].get("reasoning", "")
            print("\nReasoning preview:")
            print(str(first_reasoning)[:800])
    return sample


In [10]:
#sample = run_query(query, num_iters=num_iters)


In [11]:
#print_top_predictions(sample, k=5)


## ECtHR Multi-Case Article Retrieval Test

This section loads ECtHR cases, joins each case's fact paragraphs into one LATTICE query, retrieves Convention/Protocol article leaves from the EU conventions tree, and compares the predicted article set with the dataset's multi-label article targets.

The main count is `all_gold_found`: a case is counted as correct when every expected article label appears somewhere in the top retrieved article set. Because a case may involve multiple articles, the table also reports partial matches, precision, recall, and F1.


In [12]:
import re

import pandas as pd
from datasets import load_dataset


def load_ecthr_dataset(split: str = "train", config: str = "alleged-violation-prediction"):
    """Load ECtHR Cases directly from Hugging Face.

    `alleged-violation-prediction` is the task that matches this experiment:
    given the case facts, predict the Convention articles that may be related
    to / alleged in the case. Use `violation-prediction` if you instead want
    the articles the Court found violated.

    The dataset uses a loading script, so `trust_remote_code=True` is required.
    """
    dataset = load_dataset(
        "AUEB-NLP/ecthr_cases",
        config,
        split=split,
        trust_remote_code=True,
    )
    return dataset


def get_label_names(dataset, label_column: str = "labels") -> list[str] | None:
    """No label-name lookup is needed for `AUEB-NLP/ecthr_cases`.

    The dataset already stores labels as article strings, e.g. `"6"` or
    `"P1-1"`. The argument is kept so the rest of the notebook stays generic.
    """
    return None


def normalize_article_label(value) -> str | None:
    """Normalize labels from ECtHR Cases or article text.

    Examples:
    - 6 -> article_6
    - "Article 6" -> article_6
    - "P1-1" -> protocol_1_article_1
    - "article_1_of_protocol_1" -> protocol_1_article_1
    """
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None

    s = s.replace("no violation", "")
    s = s.replace("non violation", "")
    s = s.replace("non-violation", "")
    s = s.replace(".", "")
    s = s.replace("-", "_")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"^echr_", "", s)
    s = re.sub(r"^convention_", "", s)

    # ECtHR Cases labels bare Convention articles as strings: "2", "3", "6"...
    if re.fullmatch(r"\d+[a-z]?", s):
        return f"article_{s}"

    # Compact protocol labels appear as P1-1, P4-2, etc. after normalization: p1_1.
    m = re.fullmatch(r"p(\d+)_(\d+)", s)
    if m:
        return f"protocol_{int(m.group(1))}_article_{int(m.group(2))}"

    # article_1_of_protocol_1 / article_1_protocol_1
    m = re.search(r"article_?(\d+[a-z]?)_(?:of_)?protocol_?(\d+)", s)
    if m:
        return f"protocol_{int(m.group(2))}_article_{m.group(1)}"

    # protocol_1_article_1
    m = re.search(r"protocol_?(\d+)_article_?(\d+[a-z]?)", s)
    if m:
        return f"protocol_{int(m.group(1))}_article_{m.group(2)}"

    m = re.search(r"article_?(\d+[a-z]?)", s)
    if m:
        return f"article_{m.group(1)}"

    return None


def example_gold_articles(example: dict, label_names: list[str] | None = None, label_column: str = "labels") -> set[str]:
    """Convert one ECtHR example's labels to normalized article IDs."""
    raw_labels = example.get(label_column, [])
    if raw_labels is None:
        return set()
    if isinstance(raw_labels, (int, str)):
        raw_labels = [raw_labels]

    gold = set()
    for raw_label in raw_labels:
        label_value = label_names[raw_label] if label_names and isinstance(raw_label, int) else raw_label
        normalized = normalize_article_label(label_value)
        if normalized:
            gold.add(normalized)
    return gold



In [18]:
TREE_ARTICLE_RE = re.compile(
    r"(?:Protocol\s+(?P<protocol>\d+)\s+)?Art\.\s*(?P<article>\d+[A-Za-z]?)",
    flags=re.IGNORECASE,
)


def facts_to_case_prompt(facts: list[str], max_chars: int = 12000) -> str:
    """Join ECtHR fact paragraphs into one retrieval query for LATTICE."""
    fact_text = "\n".join(f"- {fact.strip()}" for fact in facts if str(fact).strip())
    if len(fact_text) > max_chars:
        fact_text = fact_text[:max_chars] + "\n- [facts truncated]"

    return f"""You are an LLM traversing a semantic knowledge tree of European Convention law.

The tree is structured as follows:
- Internal nodes are semantic summaries of legal concepts or clusters of rights.
- Each internal node summarizes the information contained in its child nodes.
- Leaf nodes are specific Convention or Protocol articles and their provisions.

Given the facts of an ECtHR case, traverse the tree from the root toward the most legally relevant leaf articles.

Your task is not to retrieve every article that is semantically related. Your task is to identify the articles most likely to be used in an application before the European Court of Human Rights.

Traversal rules:
1. First identify the central legal injury in the facts.
2. Move only into child nodes whose summary directly matches that injury.
3. Penalize branches that are only factually mentioned, background-related, or speculative.
4. Prefer branches where the facts show an actual state interference, omission, procedural defect, or lack of remedy.
5. Continue traversal until you reach the strongest article leaves.
6. Return only leaf articles whose full path from the root is strongly supported by the facts.

For each selected article leaf:
- Give the article number and title.
- Explain the semantic path that led to it.
- Explain why it is a primary or secondary article.
- Retrieve the relevant provision text.
- Assign a relevance score from 0 to 1.

Do not include weak articles unless they are necessary to explain why they were rejected.

Case facts:
{fact_text}
"""


def extract_articles_from_tree_text(text: str) -> set[str]:
    """Extract normalized article IDs from retrieved EU conventions tree leaves."""
    predictions = set()
    for match in TREE_ARTICLE_RE.finditer(text or ""):
        article = match.group("article").lower()
        protocol = match.group("protocol")
        if protocol:
            predictions.add(f"protocol_{int(protocol)}_article_{article}")
        else:
            predictions.add(f"article_{article}")
    return predictions


def predicted_articles_from_sample(sample: InferSample, k: int = 10) -> tuple[set[str], list[dict]]:
    """Take top LATTICE leaf predictions and extract article IDs from their text."""
    predicted = set()
    rows = []
    for rank, (node, score) in enumerate(sample.get_top_predictions(k=k), start=1):
        article_ids = extract_articles_from_tree_text(node.desc)
        predicted |= article_ids
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "node_id": node.id,
                "articles": sorted(article_ids),
                "text": node.desc[:800],
            }
        )
    return predicted, rows


def score_prediction(gold: set[str], predicted: set[str]) -> dict:
    """Score one multi-label case."""
    true_positive = len(gold & predicted)
    precision = true_positive / len(predicted) if predicted else 0.0
    recall = true_positive / len(gold) if gold else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    return {
        "any_gold_found": bool(gold & predicted),
        "all_gold_found": bool(gold) and gold.issubset(predicted),
        "exact_set_match": bool(gold) and gold == predicted,
        "true_positive": true_positive,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_ecthr_case(example: dict, label_names: list[str] | None, *, num_iters: int = 6, top_k: int = 10):
    facts = example.get("text") or example.get("facts") or []
    if isinstance(facts, str):
        facts = [facts]

    query = facts_to_case_prompt(facts)
    sample = run_query(query, num_iters=num_iters)
    predicted, top_rows = predicted_articles_from_sample(sample, k=top_k)
    gold = example_gold_articles(example, label_names)
    metrics = score_prediction(gold, predicted)

    return {
        "query": query,
        "gold": sorted(gold),
        "predicted": sorted(predicted),
        "top_rows": top_rows,
        "sample": sample,
        **metrics,
    }


def evaluate_ecthr_cases(dataset, label_names: list[str] | None, *, n_cases: int = 10, num_iters: int = 6, top_k: int = 10, start: int = 0):
    """Loop over multiple ECtHR cases and count how often LATTICE finds the expected articles."""
    results = []
    for local_idx, example in enumerate(dataset.select(range(start, min(start + n_cases, len(dataset))))):
        case_idx = start + local_idx
        print(f"\n================ ECtHR case {case_idx} ================")
        result = run_ecthr_case(example, label_names, num_iters=num_iters, top_k=top_k)
        result["case_index"] = case_idx
        results.append(result)
        print("Gold:     ", result["gold"])
        print("Predicted: ", result["predicted"])
        print(
            f"Correct(all gold found): {result['all_gold_found']} | "
            f"Recall: {result['recall']:.2f} | Precision: {result['precision']:.2f} | F1: {result['f1']:.2f}"
        )

    df = pd.DataFrame(
        [
            {
                "case_index": r["case_index"],
                "gold": r["gold"],
                "predicted": r["predicted"],
                "any_gold_found": r["any_gold_found"],
                "all_gold_found": r["all_gold_found"],
                "exact_set_match": r["exact_set_match"],
                "true_positive": r["true_positive"],
                "precision": r["precision"],
                "recall": r["recall"],
                "f1": r["f1"],
            }
            for r in results
        ]
    )
    print("\n================ Summary ================")
    print(f"Cases evaluated: {len(df)}")
    print(f"Any gold article found: {int(df['any_gold_found'].sum())}/{len(df)}")
    print(f"All gold articles found: {int(df['all_gold_found'].sum())}/{len(df)}")
    print(f"Exact set match: {int(df['exact_set_match'].sum())}/{len(df)}")
    print(f"Mean recall: {df['recall'].mean():.3f}")
    print(f"Mean precision: {df['precision'].mean():.3f}")
    print(f"Mean F1: {df['f1'].mean():.3f}")
    return df, results



In [ ]:
# Configure the ECtHR evaluation run.
# Use split="train" if you want to test quickly on the training set facts,
# or split="test" for the held-out ECtHR Cases test split.
ECTHR_CONFIG = "alleged-violation-prediction"
ECTHR_SPLIT = "train"
N_CASES = 10
NUM_ITERS_PER_CASE = 10
TOP_K_LEAVES = 15

ecthr_dataset = load_ecthr_dataset(split=ECTHR_SPLIT, config=ECTHR_CONFIG)
ecthr_label_names = get_label_names(ecthr_dataset)

print(ecthr_dataset)
print("Label names:", ecthr_label_names)
print("First example labels:", ecthr_dataset[0]["labels"])

results_df, ecthr_results = evaluate_ecthr_cases(
    ecthr_dataset,
    ecthr_label_names,
    n_cases=N_CASES,
    num_iters=NUM_ITERS_PER_CASE,
    top_k=TOP_K_LEAVES,
)

results_df


2026-05-08 17:45:01,382 - eu_conventions_traversal_notebook - INFO - Running a batch of 1 prompts...
2026-05-08 17:45:01,388 - eu_conventions_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Dataset({
    features: ['facts', 'labels', 'silver_rationales', 'gold_rationales'],
    num_rows: 9000
})
Label names: None
First example labels: ['13', '8']

================ ECtHR case 0 ================
Running traversal for query: You are an LLM traversing a semantic knowledge tree of European Convention law.

The tree is structured as follows:
- Internal nodes are semantic summaries of legal concepts or clusters of rights.
- Each internal node summarizes the information contained in its child nodes.
- Leaf nodes are specific Convention or Protocol articles and their provisions.

Given the facts of an ECtHR case, traverse the tree from the root toward the most legally relevant leaf articles.

Your task is not to retrieve every article that is semantically related. Your task is to identify the articles most likely to be used in an application before the European Court of Human Rights.

Traversal rules:
1. First identify the central legal injury in the facts.
2. Move only into child

Processing batch:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-08 17:45:09,771 - eu_conventions_traversal_notebook - INFO - Running a batch of 3 prompts...
2026-05-08 17:45:09,774 - eu_conventions_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Current beam state paths:
  path=(2,) | path_relevance=1.000 | desc=EU Protocols: Rights, Procedures, and Committees. This subtree covers the legal rights, procedural frameworks, and institutional mechanisms established by EU Protocols and the Euro
  path=(0,) | path_relevance=0.600 | desc=ECHR Principles and EU Treaty Signing. Provides information on the foundational principles and preamble of the European Convention on Human Rights (ECHR), as well as the formal leg
  path=(1,) | path_relevance=0.050 | desc=European Union Organization and Governance. Provides information on the structure, governance, and legislative processes of the European Union, including details about its main ins

Reasoning preview:
1. Essential Problem: The facts describe a situation where state authorities intervened heavily in a family's private life, especially relating to child custody, parental access, and the removal of children from their parents due to concerns about the mother's mental health. The prima

...

Current beam state paths:
  path=(0, 0) | path_relevance=0.325 | desc=ECHR Preamble and Principles. Covers the preamble and foundational principles of the European Convention on Human Rights (ECHR), addressing its objectives, philosophical underpinni
  path=(2, 0) | path_relevance=0.050 | desc=EU Protocols: Rights, Procedures, and Governance. This subtree addresses questions about the rights and freedoms protected by EU Protocols, the legal and procedural framework for t
  path=(0, 1) | path_relevance=0.000 | desc=EU Treaty Signing Procedures. Covers the formal processes, legal requirements, and official authorization steps for signing EU treaties, including the necessary legal language and 
  path=(1, 0) | path_relevance=0.000 | desc=European Union Structure and Governance. Covers foundational aspects of the European Union, including its organizational structure, member states, main institutions, governance sys
  path=(2, 1) | path_relevance=0.000 | desc=EU Convention Emergency Powers

Processing batch:   0%|          | 0/8 [00:00<?, ?it/s]

2026-05-08 18:00:01,404 - eu_conventions_traversal_notebook - INFO - Running a batch of 8 prompts...
2026-05-08 18:00:01,407 - eu_conventions_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Current beam state paths:
  path=(2, 0, 1) | path_relevance=0.525 | desc=ECHR/EU Protocol Court Procedures and Structure. This subtree provides detailed information on the procedural, structural, and institutional aspects of the European Court of Human 
  path=(2, 0, 0) | path_relevance=0.350 | desc=EU Protocols: Rights, Applications, and Procedures. This subtree addresses questions about the rights and freedoms protected by EU Protocols, the interpretation and application of 
  path=(2, 0, 2) | path_relevance=0.075 | desc=EU Protocols: Legal Governance. This subtree provides information on the legal governance of EU protocols, covering the frameworks, procedures, and obligations related to protocol 
  path=(0, 0, 0) | path_relevance=0.050 | desc=ECHR Preamble and Foundations. Provides information on the preamble and foundational principles of the European Convention on Human Rights (ECHR), including its objectives, philoso
  path=(2, 0, 3) | path_relevance=0.050 | desc=EU Convention W

Processing batch:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
ecthr_dataset[0]

In [15]:
ecthr_results

[{'query': 'You are given the facts of a European Court of Human Rights case.\nFind the Convention or Protocol articles that might be legally related to this case.\nThere may be multiple relevant articles. Retrieve article provisions that could apply.\n\nCase facts:\n- 11.  At the beginning of the events relevant to the application, K. had a daughter, P., and a son, M., born in 1986 and 1988 respectively. P.’s father is X and M.’s father is V. From March to May 1989 K. was voluntarily hospitalised for about three months, having been diagnosed as suffering from schizophrenia. From August to November 1989 and from December 1989 to March 1990, she was again hospitalised for periods of about three months on account of this illness. In 1991 she was hospitalised for less than a week, diagnosed as suffering from an atypical and undefinable psychosis. It appears that social welfare and health authorities have been in contact with the family since 1989.\n- 12.  The applicants initially cohabite

In [16]:
# Optional: visualize the prediction tree explored by the LLM.
VIS_DIR = REPO_ROOT / "results" / "tree_construction"
VIS_DIR.mkdir(parents=True, exist_ok=True)
html_path = VIS_DIR / "eu_conventions_traversal_prediction_tree.html"

fig = visualize_sample(sample, width=1400, height=900, save_path=str(html_path))
print(html_path)


NameError: name 'sample' is not defined

## Notes

- `num_iters` controls how deep the traversal goes before you inspect results.
- If `print_top_predictions(...)` says there are no leaf predictions yet, increase `num_iters`.
- `hp.SUBSET` controls the relevance definition used inside the traversal prompt. For legal QA-style usage, `fiqa` is usually a reasonable default here.
- `visualize_sample(...)` shows the explored prediction tree, not the static semantic tree structure.
- The default tree path first checks `trees/EU/eu_conventions_notebook/tree-bottom-up-llm.pkl`, then falls back to the existing `trees/EU/codigo_civil_notebook/tree-bottom-up-llm.pkl` path.

- The ECtHR section uses `AUEB-NLP/ecthr_cases` (`alleged-violation-prediction` by default). The case facts are joined into one query and the label list is treated as the expected multi-article target set.
